# 🚕 NYC Yellow Taxi Trip Data
## Dataset: yellow_tripdata_2016-03.csv

In [16]:
import pandas as pd
from pathlib import Path

### 1. Extract - Leitura dos Dados

In [17]:
df = pd.read_csv('../data/input/yellow_tripdata_2016-03.csv', nrows=500000)

print('=== Dados Carregados ===')
print(f'Total de Linhas: {len(df):,}')
print(f'Total de Colunas: {len(df.columns)}')
df.head()

=== Dados Carregados ===
Total de Linhas: 500,000
Total de Colunas: 19


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,pickup_longitude,pickup_latitude,RatecodeID,store_and_fwd_flag,dropoff_longitude,dropoff_latitude,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount
0,1,2016-03-01 00:00:00,2016-03-01 00:07:55,1,2.50,-73.98,40.77,1,N,-74.00,40.75,1,9.00,0.50,0.50,2.05,0.00,0.30,12.35
1,1,2016-03-01 00:00:00,2016-03-01 00:11:06,1,2.90,-73.98,40.77,1,N,-74.01,40.73,1,11.00,0.50,0.50,3.05,0.00,0.30,15.35
2,2,2016-03-01 00:00:00,2016-03-01 00:31:06,2,19.98,-73.78,40.64,1,N,-73.97,40.68,1,54.50,0.50,0.50,8.00,0.00,0.30,63.80
3,2,2016-03-01 00:00:00,2016-03-01 00:00:00,3,10.78,-73.86,40.77,1,N,-73.97,40.76,1,31.50,0.00,0.50,3.78,5.54,0.30,41.62
4,2,2016-03-01 00:00:00,2016-03-01 00:00:00,5,30.43,-73.97,40.79,3,N,-74.18,40.70,1,98.00,0.00,0.00,0.00,15.50,0.30,113.80


In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 19 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   VendorID               500000 non-null  int64  
 1   tpep_pickup_datetime   500000 non-null  object 
 2   tpep_dropoff_datetime  500000 non-null  object 
 3   passenger_count        500000 non-null  int64  
 4   trip_distance          500000 non-null  float64
 5   pickup_longitude       500000 non-null  float64
 6   pickup_latitude        500000 non-null  float64
 7   RatecodeID             500000 non-null  int64  
 8   store_and_fwd_flag     500000 non-null  object 
 9   dropoff_longitude      500000 non-null  float64
 10  dropoff_latitude       500000 non-null  float64
 11  payment_type           500000 non-null  int64  
 12  fare_amount            500000 non-null  float64
 13  extra                  500000 non-null  float64
 14  mta_tax                500000 non-nu

In [19]:
# visualizar o describe sem a notacao cientifica
pd.set_option('display.float_format', '{:.2f}'.format)

print(df.describe())

       VendorID  passenger_count  trip_distance  pickup_longitude  \
count 500000.00        500000.00      500000.00         500000.00   
mean       1.62             1.70          12.88            -73.01   
std        0.48             1.39        7071.06              8.38   
min        1.00             0.00           0.00           -121.93   
25%        1.00             1.00           1.00            -73.99   
50%        2.00             1.00           1.64            -73.98   
75%        2.00             2.00           3.02            -73.97   
max        2.00             8.00     5000000.00              0.00   

       pickup_latitude  RatecodeID  dropoff_longitude  dropoff_latitude  \
count        500000.00   500000.00          500000.00         500000.00   
mean             40.22        1.04             -73.06             40.25   
std               4.62        0.50               8.17              4.50   
min               0.00        1.00            -121.93              0.00   
25%

### 2. Data Quality & Anomalias

In [20]:
# converter colunas de data
df['pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])
df['dropoff_datetime'] = pd.to_datetime(df['tpep_dropoff_datetime'])

# verificar valores nulos
print('=== Valores Nulos ===')
print(df.isnull().sum())

=== Valores Nulos ===
VendorID                 0
tpep_pickup_datetime     0
tpep_dropoff_datetime    0
passenger_count          0
trip_distance            0
pickup_longitude         0
pickup_latitude          0
RatecodeID               0
store_and_fwd_flag       0
dropoff_longitude        0
dropoff_latitude         0
payment_type             0
fare_amount              0
extra                    0
mta_tax                  0
tip_amount               0
tolls_amount             0
improvement_surcharge    0
total_amount             0
pickup_datetime          0
dropoff_datetime         0
dtype: int64


In [21]:
total = len(df)

# numero de passageiros inválidos (<=0 ou >4)
invalid_passenger_count = ((df['passenger_count'] <= 0) | (df['passenger_count'] > 4))

# numero de viagens com distância inválida (<=0 ou >100) (milhas)
invalid_trip_distance = ((df['trip_distance'] <= 0) | (df['trip_distance'] > 100))

# numero de viagens com RatecodeID inválido (>6)
invalid_ratecodid = (df['RatecodeID'] > 6)

# numero de viagens com tarifa negativa
invalid_tip_amount = (df['tip_amount'] < 0)

# numero de viagens com duração inválida (pickup_datetime > dropoff_datetime)
invalid_duration = (df['pickup_datetime'] > df['dropoff_datetime'])

# numero de viagens com coordenadas inválidas (pickup e dropoff fora de NYC)
invalid_pickup_coords = (
    (df['pickup_latitude'] < 40.4) | (df['pickup_latitude'] > 41.0) |
    (df['pickup_longitude'] < -74.3) | (df['pickup_longitude'] > -73.7)
)
invalid_dropoff_coords = (
    (df['dropoff_latitude'] < 40.4) | (df['dropoff_latitude'] > 41.0) |
    (df['dropoff_longitude'] < -74.3) | (df['dropoff_longitude'] > -73.7)
)

print('=== Anomalias Encontradas ===')

print(f'Passageiros Invalidos (<=0 ou >4):  {invalid_passenger_count.sum():,} ({invalid_passenger_count.sum()/total*100:.2f}%)')
print(f'Distancia Invalida (<=0 ou >100): {invalid_trip_distance.sum():,} ({invalid_trip_distance.sum()/total*100:.2f}%)')
print(f'RatecodeID Invalido (>6):           {invalid_ratecodid.sum():,} ({invalid_ratecodid.sum()/total*100:.3f}%)')
print(f'Gorjetas Invalidas:                 {invalid_tip_amount.sum():,} ({invalid_tip_amount.sum()/total*100:.2f}%)')
print(f'Duracao Invalida:                   {invalid_duration.sum():,} ({invalid_duration.sum()/total*100:.3f}%)')
print(f'Coordenadas Pickup Fora de NYC:     {invalid_pickup_coords.sum():,} ({invalid_pickup_coords.sum()/total*100:.2f}%)')
print(f'Coordenadas Dropoff Fora de NYC:    {invalid_dropoff_coords.sum():,} ({invalid_dropoff_coords.sum()/total*100:.2f}%)')

=== Anomalias Encontradas ===
Passageiros Invalidos (<=0 ou >4):  50,873 (10.17%)
Distancia Invalida (<=0 ou >100): 2,793 (0.56%)
RatecodeID Invalido (>6):           9 (0.002%)
Gorjetas Invalidas:                 5 (0.00%)
Duracao Invalida:                   2 (0.000%)
Coordenadas Pickup Fora de NYC:     6,578 (1.32%)
Coordenadas Dropoff Fora de NYC:    6,506 (1.30%)


In [22]:
# criar máscara para identificar linhas inválidas
invalid_mask = (
    invalid_passenger_count |
    invalid_trip_distance |
    invalid_ratecodid |
    invalid_tip_amount |
    invalid_duration |
    invalid_pickup_coords |
    invalid_dropoff_coords
)

# remover linhas inválidas e colunas
df_cleaned = df.drop(df[invalid_mask].index)
df_cleaned = df_cleaned.drop(columns=['tpep_pickup_datetime', 'tpep_dropoff_datetime'])

removed_rows = total - len(df_cleaned)
print(f'Total de Linhas Antes da Remocao: {total:,}')
print(f'Total de Linhas Removidas: {removed_rows:,} ({removed_rows/total*100:.2f}%)')
print(f'Total de Linhas Apos a Remocao: {len(df_cleaned):,}')

Total de Linhas Antes da Remocao: 500,000
Total de Linhas Removidas: 59,542 (11.91%)
Total de Linhas Apos a Remocao: 440,458


### 3. Transform - Colunas Calculadas

In [23]:
# duracao da viagem em minutos
df_cleaned['trip_duration_min'] = (df_cleaned['dropoff_datetime'] - df_cleaned['pickup_datetime']).dt.total_seconds() / 60

# hora do dia
df_cleaned['hour_of_day'] = df_cleaned['pickup_datetime'].dt.hour

# data sem hora
df_cleaned['date'] = df_cleaned['pickup_datetime'].dt.date

# dia da semana
df_cleaned['day_of_week'] = df_cleaned['pickup_datetime'].dt.day_name()

# Uma forma mais segura de evitar valores infinitos (inf)
df_cleaned['trip_speed_mph'] = (df_cleaned['trip_distance'] / (df_cleaned['trip_duration_min'] / 60))
df_cleaned['trip_speed_mph'] = df_cleaned['trip_speed_mph'].fillna(0).replace(np.inf, 0)
# tarifa por milha
df_cleaned['tip_pct'] = df_cleaned['tip_amount'] / df_cleaned['fare_amount'] * 100
df_cleaned['tip_pct'] = df_cleaned['tip_pct'].fillna(0).replace(np.inf, 0)

# receita por milha
df_cleaned['revenue_per_mile'] = (df_cleaned['total_amount'] / df_cleaned['trip_distance'])
df_cleaned['revenue_per_mile'] = df_cleaned['revenue_per_mile'].fillna(0).replace(np.inf, 0)

new_columns = [
    'trip_duration_min',
    'hour_of_day',
    'day_of_week',
    'date',
    'trip_speed_mph',
    'tip_pct',
    'revenue_per_mile'
]

print('=== Novas Colunas Criadas ===')
for col in new_columns:
    print(f'- {col}')

print('\nTotal de Colunas em DF', len(df.columns))
print('Total de Colunas em DF_CLENAED:', len(df_cleaned.columns))

df_cleaned[new_columns].head(10)

=== Novas Colunas Criadas ===
- trip_duration_min
- hour_of_day
- day_of_week
- date
- trip_speed_mph
- tip_pct
- revenue_per_mile

Total de Colunas em DF 21
Total de Colunas em DF_CLENAED: 26


,trip_duration_min,hour_of_day,day_of_week,date,trip_speed_mph,tip_pct,revenue_per_mile
0,7.92,0,Tuesday,2016-03-01,18.95,22.78,4.94
1,11.10,0,Tuesday,2016-03-01,15.68,27.73,5.29
2,31.10,0,Tuesday,2016-03-01,38.55,14.68,3.19
3,0.00,0,Tuesday,2016-03-01,0.00,12.00,3.86
7,16.05,0,Tuesday,2016-03-01,23.18,0.00,3.52
8,4.98,0,Tuesday,2016-03-01,8.43,36.36,12.57
9,24.08,0,Tuesday,2016-03-01,17.89,13.62,3.90
10,2.03,0,Tuesday,2016-03-01,15.93,0.00,9.81
11,7.78,0,Tuesday,2016-03-01,13.10,0.00,5.47
12,3.05,0,Tuesday,2016-03-01,21.64,40.00,8.18


In [24]:
df_cleaned['revenue_per_mile'].describe()

count   440458.00
mean         8.28
std         45.04
min      -5730.00
25%          5.30
50%          6.99
75%          9.07
max       7541.00
Name: revenue_per_mile, dtype: float64

### 4. Agregações & Métricas

In [25]:
# tabela dinâmica por hora do dia
hourly = df_cleaned.groupby('hour_of_day').agg(
    total_trips=('hour_of_day', 'size'),
    avg_distance=('trip_distance', 'mean'),
    avg_fare=('fare_amount', 'mean'),
    avg_duration=('trip_duration_min', 'mean'),
    avg_speed=('trip_speed_mph', 'mean'),
    avg_tip_per=('tip_pct', 'mean'),
)

print('=== Tabela Dinamica por Hora do Dia ===')
print(hourly)

=== Tabela Dinamica por Hora do Dia ===
             total_trips  avg_distance  avg_fare  avg_duration  avg_speed  \
hour_of_day                                                                 
0                  14483          3.78     13.68         13.82      23.39   
1                   8514          3.54     12.86         13.37      18.53   
2                   5530          3.44     12.70         12.40      19.32   
3                   3651          3.69     13.41         15.43      19.35   
4                   3604          4.63     15.70         13.19      25.30   
5                   6944          4.04     14.00         12.15      20.53   
6                   9332          3.09     11.69         11.76      16.07   
7                  24640          2.72     11.70         14.63      12.46   
8                  29027          2.44     11.88         16.39      10.45   
9                  26582          2.51     12.18         16.13       9.66   
10                 23023          2.

In [26]:
# tabela dinamica por vendor
vendor = df_cleaned.groupby('VendorID').agg(
    total_trips=('VendorID', 'size'),
    avg_distance=('trip_distance', 'mean'),
    avg_fare=('fare_amount', 'mean'),
    total_revenue=('total_amount', 'sum'),
    avg_tip_per=('tip_pct', 'mean'),
    avg_speed=('trip_speed_mph', 'mean')
)

print('=== Tabela Dinamica por Vendor ===')
print(vendor)

=== Tabela Dinamica por Vendor ===
          total_trips  avg_distance  avg_fare  total_revenue  avg_tip_per  \
VendorID                                                                    
1              182610          2.83     12.24     2832902.88        14.79   
2              257848          2.92     12.75     4139361.11        14.57   

          avg_speed  
VendorID             
1             15.61  
2             11.42  


In [27]:
payment_labels = {
    1: 'Credit Card',
    2: 'Cash',
    3: 'No Charge',
    4: 'Dispute',
}

# mapear os valores de payment_type para o payment_labels
df_cleaned['payment_type'] = df_cleaned['payment_type'].map(payment_labels)

# tabela dinamica por tipo de pagamento
payment = df_cleaned.groupby('payment_type').agg(
    total_trips=('payment_type', 'size'),
    avg_fare=('fare_amount', 'mean'),
    avg_tip=('tip_amount', 'mean'),
    avg_tip_per=('tip_pct', 'mean'),
    total_revenue=('total_amount', 'sum'),
)

payment['pct_trips'] = ((payment['total_trips'] / payment['total_trips'].sum()) * 100).round(2)

print('=== Tabela Dinamica por Tipo de Pagamento ===')
print(payment)

=== Tabela Dinamica por Tipo de Pagamento ===
              total_trips  avg_fare  avg_tip  avg_tip_per  total_revenue  \
payment_type                                                               
Cash               139215     11.44     0.00         0.00     1779492.65   
Credit Card        299700     13.05     2.69        21.54     5170791.00   
Dispute               455     10.83     0.00         0.02        5432.14   
No Charge            1088     12.94     0.01         0.02       16548.20   

              pct_trips  
payment_type             
Cash              31.61  
Credit Card       68.04  
Dispute            0.10  
No Charge          0.25  


In [28]:
# tabela dinamica por dia da semana
weekday = df_cleaned.groupby('day_of_week').agg(
    total_trips=('day_of_week', 'size'),
    avg_distance=('trip_distance', 'mean'),
    avg_fare=('fare_amount', 'mean'),
    avg_duration=('trip_duration_min', 'mean'),
    avg_tip_per=('tip_pct', 'mean'),
)

# reorganizar os dias da semana na ordem correta
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
weekday = weekday.reindex(day_order)

print('=== Tabela Dinamica por Dia da Semana ===')
print(weekday)

=== Tabela Dinamica por Dia da Semana ===
             total_trips  avg_distance  avg_fare  avg_duration  avg_tip_per
day_of_week                                                                
Monday               NaN           NaN       NaN           NaN          NaN
Tuesday        333900.00          2.83     12.30         14.81        14.84
Wednesday       23714.00          3.68     13.35         13.56        14.06
Thursday        82844.00          2.87     13.27         18.77        14.08
Friday               NaN           NaN       NaN           NaN          NaN
Saturday             NaN           NaN       NaN           NaN          NaN
Sunday               NaN           NaN       NaN           NaN          NaN


### 5. Load - Parquet

In [29]:
# converter a coluna 'date' para string antes de salvar em parquet
df_cleaned['date'] = df_cleaned['date'].astype(str)

# salvar o dataframe limpo em formato parquet
output_path = Path('../data/output/yellow_tripdata_2016-03_cleaned.parquet')
output_path.parent.mkdir(parents=True, exist_ok=True)
df_cleaned.to_parquet(output_path, index=False)

print('=== Parquet ===')
print(f'Dataframe Limpo Salvo em: {output_path}')
print(f'Tamanho Total: {output_path.stat().st_size / (1024 * 1024):.2f} MB')

=== Parquet ===
Dataframe Limpo Salvo em: ..\data\output\yellow_tripdata_2016-03_cleaned.parquet
Tamanho Total: 14.71 MB


In [30]:
df_parquet = pd.read_parquet(output_path)
print('=== Parquet Carregado ===')
print(f'Total de Linhas em PARQUET: {len(df_parquet):,}')
print(f'Total de Linhas em DF_CLEANED: {len(df_cleaned):,}')
print(f'Math: {'MATH' if len(df_parquet) == len(df_cleaned) else 'ERROR'}')
print(f'\nColunas: {df_parquet.columns.tolist()}')

df_parquet.head(10)

=== Parquet Carregado ===
Total de Linhas em PARQUET: 440,458
Total de Linhas em DF_CLEANED: 440,458
Math: MATH

Colunas: ['VendorID', 'passenger_count', 'trip_distance', 'pickup_longitude', 'pickup_latitude', 'RatecodeID', 'store_and_fwd_flag', 'dropoff_longitude', 'dropoff_latitude', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'pickup_datetime', 'dropoff_datetime', 'trip_duration_min', 'hour_of_day', 'date', 'day_of_week', 'trip_speed_mph', 'tip_pct', 'revenue_per_mile']


,VendorID,passenger_count,trip_distance,pickup_longitude,pickup_latitude,RatecodeID,store_and_fwd_flag,dropoff_longitude,dropoff_latitude,payment_type,...,total_amount,pickup_datetime,dropoff_datetime,trip_duration_min,hour_of_day,date,day_of_week,trip_speed_mph,tip_pct,revenue_per_mile
0,1,1,2.50,-73.98,40.77,1,N,-74.00,40.75,Credit Card,...,12.35,2016-03-01 00:00:00,2016-03-01 00:07:55,7.92,0,2016-03-01,Tuesday,18.95,22.78,4.94
1,1,1,2.90,-73.98,40.77,1,N,-74.01,40.73,Credit Card,...,15.35,2016-03-01 00:00:00,2016-03-01 00:11:06,11.10,0,2016-03-01,Tuesday,15.68,27.73,5.29
2,2,2,19.98,-73.78,40.64,1,N,-73.97,40.68,Credit Card,...,63.80,2016-03-01 00:00:00,2016-03-01 00:31:06,31.10,0,2016-03-01,Tuesday,38.55,14.68,3.19
3,2,3,10.78,-73.86,40.77,1,N,-73.97,40.76,Credit Card,...,41.62,2016-03-01 00:00:00,2016-03-01 00:00:00,0.00,0,2016-03-01,Tuesday,0.00,12.00,3.86
4,1,1,6.20,-73.79,40.65,1,N,-73.83,40.71,No Charge,...,21.80,2016-03-01 00:00:01,2016-03-01 00:16:04,16.05,0,2016-03-01,Tuesday,23.18,0.00,3.52
5,1,1,0.70,-73.96,40.76,1,N,-73.97,40.76,Credit Card,...,8.80,2016-03-01 00:00:01,2016-03-01 00:05:00,4.98,0,2016-03-01,Tuesday,8.43,36.36,12.57
6,2,3,7.18,-73.99,40.74,1,N,-73.95,40.80,Credit Card,...,28.00,2016-03-01 00:00:01,2016-03-01 00:24:06,24.08,0,2016-03-01,Tuesday,17.89,13.62,3.90
7,2,2,0.54,-73.99,40.76,1,N,-73.99,40.76,Cash,...,5.30,2016-03-01 00:00:01,2016-03-01 00:02:03,2.03,0,2016-03-01,Tuesday,15.93,0.00,9.81
8,1,1,1.70,-73.97,40.80,1,N,-73.94,40.80,Cash,...,9.30,2016-03-01 00:00:02,2016-03-01 00:07:49,7.78,0,2016-03-01,Tuesday,13.10,0.00,5.47
9,1,1,1.10,-73.95,40.79,1,N,-73.97,40.80,Credit Card,...,9.00,2016-03-01 00:00:02,2016-03-01 00:03:05,3.05,0,2016-03-01,Tuesday,21.64,40.00,8.18
